# 04 — Chan doan phan du R

$R = \Delta(N)$ la phan khoang cach con lai sau khi ca bon control da bat.

**$R$ khong duoc goi la concept drift.** No la **can tren cho phan dong gop cua
drift**, vi no chua moi thu chua kiem soat: mot confound thu nam chua liet ke,
sai lech do tham so, hieu ung co mau o cua so muon, sai so nhan ho trong MLRan,
va drift.

## Quy tac dien giai, chot truoc khi nhin so lieu (muc 9)

| Dau vao doi ro | Phan phoi co dieu kien doi | Ket luan |
|---|---|---|
| co | khong co bang chung | **covariate shift**, khong phai concept drift |
| co | co | **concept drift** |

**Ket qua tren du lieu that: khong phep kiem nao trong nghien cuu nay quan sat
duoc $p(y|x)$**, nen cot thu hai luon la "khong co bang chung" va ket luan la
covariate shift. Ly do day du o muc 5b.
| khong ro | khong ro | **phan du chua giai thich duoc**, co the tu nguon ngoai taxonomy |

Va mot cong chan truoc do: **chi chay cac phep do drift o nhung o ma khoang tin
cay bootstrap cua $R$ loai tru so khong.** O nao khong loai tru thi ket luan tai
do la "phan du khong phan biet duoc voi khong" va dung lai.

Ket luan covariate shift, hoac ket luan phan du chua giai thich duoc, **deu la
ket qua tot**. Khong duoc voi lay concept drift chi vi no la cau tra loi hap dan
hon.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import warnings

import numpy as np
import pandas as pd

from sift import cache
from sift.config import ControlFlags, ExperimentConfig, PanelSpec, SplitSpec
from sift.controls import apply_controls
from sift.data import build_panel, load_mlran
from sift.drift import (
    MIN_INTERPRETABLE_HELDOUT,
    bootstrap_gap_distribution,
    conditional_structure_agreement,
    correlate_residual_with_drift,
    domain_classifier_test,
    expected_calibration_error,
)
from sift.features import feature_columns
from sift.paths import RESULTS_DIR
from sift.reporting import class_prior_reference, panel_matches_run, recompute_a1_weights
from sift.seeding import derive_seed

warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ALPHA = 0.05
N_BOOT = 1000
GATE_HALF_WIDTH = 0.10
FULL_COALITION = 15
TARGET = "ransomware_family"

## 1. Nap du lieu va kiem xuat xu

Moi thu tinh lai o duoi phu thuoc vao panel dung la panel ma lan chay da dung,
nen doi chieu van tay truoc.

In [3]:
metrics = pd.read_parquet(RESULTS_DIR / "metrics.parquet")
predictions = pd.read_parquet(RESULTS_DIR / "predictions.parquet")
panel, _ = build_panel(load_mlran(), PanelSpec())
prior_reference = class_prior_reference(panel)

manifest_path = RESULTS_DIR / "run_manifest.json"
manifest = (json.loads(manifest_path.read_text(encoding="utf-8"))
            if manifest_path.exists() else {})
provenance = panel_matches_run(panel, manifest)
print(provenance["detail"])
if not provenance["matches"]:
    raise AssertionError("panel does not match the run; nothing below can be trusted")

print(f"metrics {metrics.shape}  predictions {predictions.shape}  panel {panel.shape}")
by_fit = {name: block for name, block in predictions.groupby("fit_id", sort=False)}
print(f"prediction blocks: {len(by_fit)}")

panel matches the recorded run (0e5c02de829e5b15)
metrics (3400, 21)  predictions (722880, 6)  panel (1377, 504)
prediction blocks: 3400


## 2. Cong thu nhat: khoang tin cay bootstrap cua $R$

$R = \Delta(N) = M_{\text{reference}}(N) - M_{\text{temporal}}(N)$, tuc khoang
cach tai coalition day du.

Hai nhanh la hai lan fit rieng tren hai kho mau roi nhau nen lay mau lai doc lap
roi tru theo tung lan rut. Trong so cua A1 di kem theo tung quan sat chu khong
tinh lai tren moi lan rut, nho vay lan rut dong nhat tra ve dung diem uoc luong
da luu, va do la thu lam cho khoang nay kiem tra duoc.

Gop 5 seed vao cung mot phan phoi: bien thien theo seed cung la mot nguon bat
dinh that, khong phai thu de trung binh di.

**Cong R4: nua do rong khoang phai duoi 0,10.**

In [4]:
boot_rows = []
lattice = metrics[metrics["role"] == "lattice"]
full = lattice[lattice["config_id"] == FULL_COALITION]

for (cut, model), block in full.groupby(["cut", "model"], sort=True):
    draws = []
    for seed in sorted(block["seed"].unique()):
        arms = {}
        for design in ("random_fully_matched", "temporal"):
            row = block[(block["seed"] == seed) & (block["design"] == design)].iloc[0]
            frame = by_fit[row["fit_id"]]
            weight = (recompute_a1_weights(frame["y_true"].to_numpy(), prior_reference)
                      if bool(row["a1_prior"]) else None)
            arms[design] = (frame["y_true"].to_numpy(), frame["y_pred"].to_numpy(), weight)
        draws.append(bootstrap_gap_distribution(
            arms["random_fully_matched"], arms["temporal"],
            n_boot=N_BOOT, seed=derive_seed(0, "boot_R", int(cut), model, int(seed))))
    pooled = np.concatenate(draws)
    low, high = np.percentile(pooled, [100 * ALPHA / 2, 100 * (1 - ALPHA / 2)])
    boot_rows.append({"cut": int(cut), "model": model, "R": float(pooled.mean()),
                      "ci_low": float(low), "ci_high": float(high),
                      "half_width": float((high - low) / 2),
                      "excludes_zero": bool(low > 0 or high < 0)})
residual_ci = pd.DataFrame(boot_rows)
display(residual_ci)

,cut,model,R,ci_low,ci_high,half_width,excludes_zero
0,2015,lightgbm,0.215954,0.087799,0.336731,0.124466,True
1,2015,logreg,0.245333,0.137981,0.351916,0.106968,True
2,2015,mlp,0.220122,0.099391,0.337643,0.119126,True
3,2015,random_forest,0.193224,0.066419,0.314927,0.124254,True
4,2017,lightgbm,0.168000,-0.032589,0.366163,0.199376,False
5,2017,logreg,0.329558,0.143694,0.559979,0.208143,True
6,2017,mlp,0.267377,0.027161,0.525700,0.249270,True
7,2017,random_forest,0.242504,0.031568,0.451291,0.209862,True
8,2019,lightgbm,0.263643,0.151521,0.372858,0.110668,True
9,2019,logreg,0.293781,0.193857,0.388165,0.097154,True


In [5]:
gate_ok = bool((residual_ci["half_width"] < GATE_HALF_WIDTH).all())
n_excl = int(residual_ci["excludes_zero"].sum())
print(f"R4 gate, half-width < {GATE_HALF_WIDTH}: {gate_ok}")
print(f"  half-width range {residual_ci['half_width'].min():.4f} .. "
      f"{residual_ci['half_width'].max():.4f}")
print(f"  cells failing the gate: "
      f"{residual_ci.loc[residual_ci['half_width'] >= GATE_HALF_WIDTH, 'cut'].tolist()}")
print(f"intervals excluding zero: {n_excl} of {len(residual_ci)}")

by_cut = residual_ci.groupby("cut").agg(
    n_models=("model", "size"), R_mean=("R", "mean"),
    half_max=("half_width", "max"), n_excluding_zero=("excludes_zero", "sum"))
display(by_cut)

# A cut proceeds only if every one of its four models excludes zero. Testing a
# cut where the residual is not distinguishable from zero would be looking for
# an explanation of something not yet shown to exist.
SURVIVING = sorted(by_cut.index[by_cut["n_excluding_zero"] == by_cut["n_models"]])
STOPPED = sorted(set(by_cut.index) - set(SURVIVING))
print(f"cuts proceeding to the drift tests: {SURVIVING}")
print(f"cuts concluding at 'residual not distinguishable from zero': {STOPPED}")

R4 gate, half-width < 0.1: False
  half-width range 0.0972 .. 0.2493
  cells failing the gate: [2015, 2015, 2015, 2015, 2017, 2017, 2017, 2017, 2019, 2019, 2021, 2021, 2021, 2023, 2023, 2023, 2023]
intervals excluding zero: 17 of 20


,n_models,R_mean,half_max,n_excluding_zero
cut,,,,
2015,4,0.218658,0.124466,4
2017,4,0.251860,0.249270,3
2019,4,0.269900,0.131365,4
2021,4,0.288835,0.108181,4
2023,4,0.211791,0.239660,2


cuts proceeding to the drift tests: [2015, 2019, 2021]
cuts concluding at 'residual not distinguishable from zero': [2017, 2023]


## 3. Cua so cho cac phep do

Dung coalition co A1, A2, B2 bat va B1 tat: prior da co dinh, khong gian nhan da
gioi han, truc thoi gian dung. Do la dieu kien ma muc 9 doi hoi cho phep kiem
phan phoi co dieu kien.

Feature dung **toan bo** cot feature chu khong phai tap da chon, vi phep do o day
noi ve phan phoi dau vao; gioi han vao tap feature da chon se lam lech no.

In [6]:
COALITION = ControlFlags(a1_prior=True, a2_labels=True, b1_fs=False, b2_axis=True)
print(f"coalition index {COALITION.to_index()}  active {sorted(COALITION.active())}")

columns = feature_columns(panel)
windows = {}
for cut in sorted(metrics["cut"].unique()):
    cfg = ExperimentConfig(panel=PanelSpec(),
                           split=SplitSpec(design="temporal", cut_year=int(cut)),
                           flags=COALITION, model_name="logreg", seed=0)
    controlled = apply_controls(panel, cfg)
    windows[int(cut)] = {
        "x_early": controlled.train[columns].to_numpy(dtype=np.float64),
        "y_early": controlled.train[TARGET].to_numpy(),
        "x_late": controlled.test[columns].to_numpy(dtype=np.float64),
        "y_late": controlled.test[TARGET].to_numpy(),
    }
    print(f"cut {cut}: early {len(controlled.train):>4} x {len(columns)}  "
          f"late {len(controlled.test):>4}  "
          f"classes early {controlled.train[TARGET].nunique():>2} "
          f"late {controlled.test[TARGET].nunique():>2}")

coalition index 11  active ['a1_prior', 'a2_labels', 'b2_axis']


cut 2015: early  225 x 483  late  150  classes early  8 late  7


cut 2017: early  475 x 483  late   64  classes early 15 late  7


cut 2019: early  595 x 483  late  210  classes early 20 late 10


cut 2021: early  769 x 483  late  324  classes early 27 late 16


cut 2023: early 1317 x 483  late   60  classes early 32 late  8


## 4. Kiem phan phoi dau vao: domain classifier

Giao thuc Rabanser va cong su (2019): chia **ca hai** cua so lam doi, huan luyen
tren nua dau de phan biet source voi target, cham tren nua sau, kiem nhi thuc hai
phia voi $H_0: \text{acc} = 0.5$.

Hai diem la quyet dinh cua nhom chu khong phai cua bai bao:

- Tap giu lai duoc **can bang** bang cach lay mau bot nua lon hon. Khong can bang
  thi ty le lop da so da vuot 0,5 va gia thuyet khong sai ngay tu dau.
- Ket qua duoi **100 mau** bi danh dau khong dien giai duoc, vi chinh bai bao do
  bao method hong o vung mau it.

In [7]:
domain_rows = []
for cut, window in windows.items():
    result = domain_classifier_test(window["x_early"], window["x_late"],
                                    seed=derive_seed(0, "domain", cut))
    domain_rows.append({
        "cut": cut, "in_scope": cut in SURVIVING,
        "accuracy": result.accuracy, "excess": result.excess_accuracy,
        "n_heldout": result.n_heldout, "p_value": result.p_value,
        "interpretable": result.interpretable, "verdict": result.verdict(ALPHA)})
domain = pd.DataFrame(domain_rows)
display(domain)

for row in domain.itertuples():
    if not row.interpretable:
        print(f"cut {row.cut}: held-out {row.n_heldout} < {MIN_INTERPRETABLE_HELDOUT}, "
              "reported as undetermined, not as a p-value")

,cut,in_scope,accuracy,excess,n_heldout,p_value,interpretable,verdict
0,2015,True,0.800000,0.300000,150,5.973732e-14,True,shift
1,2017,False,0.906250,0.406250,64,9.029019e-12,False,undetermined
2,2019,True,0.804762,0.304762,210,1.183392e-19,True,shift
3,2021,True,0.839506,0.339506,324,4.172423e-37,True,shift
4,2023,False,0.833333,0.333333,60,1.616381e-07,False,undetermined


cut 2017: held-out 64 < 100, reported as undetermined, not as a p-value
cut 2023: held-out 60 < 100, reported as undetermined, not as a p-value


## 5. Kiem phan phoi co dieu kien

Domain classifier chi nhin $p(x)$. No **khong** tach duoc covariate shift voi
label shift va **khong** nhin thay $p(y|x)$, nen tu no khong bao gio du de ket
luan concept drift. Day la ve con lai.

So sanh cau truc tuong quan feature-nhan giua hai cua so, tren tap nhan chung va
voi prior da co dinh. Kem theo la **moc so sanh trong cung cua so**: hai nua roi
nhau cua cua so som, cat ve dung co cua so muon, cho biet muc dong thuan dat duoc
khi khong co gi thay doi ca. Khong co moc nay thi con so across khong doc duoc,
vi o co mau nay nhieu lay mau rat lon.

In [ ]:
# n_repeats is passed explicitly rather than left to the default: the decision
# rule reads the 2.5th percentile of the within-window null, and at twenty draws
# that percentile is essentially the minimum of the sample, unstable to tenths
# between runs of the same cut. Five hundred draws resolve it; the whole loop
# costs well under a minute.
N_COND_REPEATS = 500

cond_rows = []
for cut, window in windows.items():
    try:
        outcome = conditional_structure_agreement(
            window["x_early"], window["y_early"], window["x_late"], window["y_late"],
            seed=derive_seed(0, "conditional", cut), n_repeats=N_COND_REPEATS)
    except ValueError as error:
        cond_rows.append({"cut": cut, "in_scope": cut in SURVIVING,
                          "across": np.nan, "across_matched": np.nan,
                          "below_within_null": False, "within_mean": np.nan,
                          "within_lo": np.nan, "within_hi": np.nan,
                          "ratio": np.nan, "n_shared_classes": 0,
                          "n_features": 0, "n_late": 0,
                          "n_within_draws": 0, "n_within_at_or_below_across": 0,
                          "p_within_empirical": np.nan, "n_matched_draws": 0,
                          "n_repeats_requested": N_COND_REPEATS,
                          "note": str(error)[:60]})
        continue
    cond_rows.append({"cut": cut, "in_scope": cut in SURVIVING, **outcome,
                      "n_repeats_requested": N_COND_REPEATS, "note": ""})
conditional = pd.DataFrame(cond_rows)
display(conditional)

print("across            = agreement between the early and the late window")
print("across_matched    = the same, with the early window resampled to the late")
print("                    window's class proportions, so a change in class")
print("                    composition alone cannot depress it")
print("within            = agreement between two halves of the early window,")
print("                    cut to the size of the late one: the ceiling when")
print("                    nothing has changed")
print("below_within_null = across falls under the 2.5th percentile of that")
print("                    control. This is the decision rule: a comparison")
print("                    against a null, not against a threshold anyone chose.")
print()
print(f"null size: {N_COND_REPEATS} draws requested per cut; n_within_draws is how")
print("many were actually collected, since a split leaving fewer than two classes")
print("on a side is skipped rather than redrawn.")
for row in conditional.itertuples():
    if row.n_within_draws == 0:
        continue
    print(f"cut {row.cut}: {row.n_within_at_or_below_across} of {row.n_within_draws} "
          f"within-window draws fell at or below across={row.across:.4f}, "
          f"empirical p <= {row.p_within_empirical:.4f}")
print("p_within_empirical = (k + 1) / (N + 1), so its floor of "
      f"{1 / (N_COND_REPEATS + 1):.4f} is an upper bound, not an exact p-value.")


## 5b. Vi sao KHONG goi day la concept drift

Thong ke o muc 5 **that su phat hien mot thay doi**. E4 dung mot permutation null
phan tang giu nguyen co cua so va so mau tung lop, 500 hoan vi moi cut, va
`across` nam duoi phan vi 2,5 o **moi** cut voi **0 trong 500** hoan vi cham toi.
Prior shift khong tao ra duoc no. Attenuation khong tao ra duoc no. $p = 0{,}000$.

Nhung E4 dung tiep mot phan vi du thay vi tranh luan:

- sau feature kich hoat, moi mau bat dung mot cai, nhan bang chi so cua cai duoc bat;
- nhu vay $p(y|x)$ la mot phep doc tat dinh tu $x$, **giong het nhau o ca hai cua so theo cau tao**;
- cac feature ve tinh duoc gan lai sang lop khac o cua so muon.

Do la **covariate shift thuan tuy, $p(y|x)$ khong he doi**. Phep kiem cua chung ta
van bao "concept drift" o muc gan lai 25, 50 va 100 phan tram, voi ratio 0,85 /
0,47 / 0,09 — **bao trum khoang 0,016 den 0,62 do duoc tren du lieu that**. Trong
moi truong hop bi bao, mot classifier doc $x$ van khoi phuc $y$ hoan hao o ca hai
cua so va accuracy khong he giam.

**Ket luan.** Thong ke nay do do **on dinh cua lien ket bien giua feature va nhan**,
tuc mot tom tat cua $p(x|y)$. Ma $p(x|y)$ doi duoi covariate shift y het nhu duoi
concept drift. Phat hien khong mong manh; **tu ngu moi la sai**. Vi vay cot
`conditional_shift` bang "no evidence" theo cau tao, va ket luan la **covariate
shift**, dung nhanh ma quy tac chot truoc da dinh san.

**Duong di de gianh duoc tu "concept drift" trong nghien cuu sau.** Importance-weighted
risk: uoc luong $w(x) = p_{	ext{late}}(x)/p_{	ext{early}}(x)$ tu chinh domain
classifier da co, fit lai tren cua so som da tai trong so, roi danh gia tren cua so
muon. Neu rui ro da tai trong so van cao thi phan chenh khong quy duoc ve $p(x)$.
Canh bao cua E4: o muc phan tach 0,80 den 0,84 thi trong so rat cuc doan va khoang
tin cay nhieu kha nang qua rong de ket luan. Khong chay dem nay; mot phep kiem
thieu luc se khong doi duoc ket luan.

## 6. Calibration, chi la boi canh mo ta

Bao cao kem nhung **khong mang trong luong suy luan nao**. Ovadia va cong su cho
thay calibration suy giam ngay ca khi chi co dich chuyen dau vao, nen ECE xau di
la nhat quan voi covariate shift va khong the chong do cho ket luan concept drift.
Cau nao lap luan rang R la drift that thi khong duoc dan ECE.

ECE lech len rat manh khi moi bin it mau, nen **in kem so mau tung bin**.

Mo hinh o day la mot lan fit lai rieng de lay xac suat, vi bang prediction chi luu
nhan du doan chu khong luu xac suat. No khong phai lan fit da sinh ra ket qua.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

calibration_rows, calibration_bins = [], {}
for cut in SURVIVING:
    window = windows[cut]
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=2000, random_state=0))
    model.fit(window["x_early"], window["y_early"])
    shared = np.intersect1d(model.classes_, np.unique(window["y_late"]))
    mask = np.isin(window["y_late"], shared)
    probabilities = model.predict_proba(window["x_late"][mask])
    codes = np.searchsorted(model.classes_, window["y_late"][mask])
    ece, bins = expected_calibration_error(probabilities, codes)
    calibration_rows.append({"cut": cut, "ece": ece, "n_scored": int(mask.sum()),
                             "n_bins_occupied": len(bins),
                             "min_bin_n": int(bins["n"].min()) if len(bins) else 0})
    calibration_bins[cut] = bins
display(pd.DataFrame(calibration_rows))
for cut, bins in calibration_bins.items():
    print(f"cut {cut} bins:")
    print(bins.to_string(index=False))

,cut,ece,n_scored,n_bins_occupied,min_bin_n
0,2015,0.703255,150,8,1
1,2019,0.383817,210,8,8
2,2021,0.339714,324,9,3


cut 2015 bins:
      bin  n  confidence  accuracy
[0.2,0.3)  1    0.293640  0.000000
[0.3,0.4)  2    0.344675  0.000000
[0.4,0.5)  9    0.473913  0.222222
[0.5,0.6) 20    0.554457  0.050000
[0.6,0.7) 10    0.642581  0.000000
[0.7,0.8) 16    0.754835  0.062500
[0.8,0.9) 22    0.846650  0.045455
[0.9,1.0) 70    0.971734  0.157143
cut 2019 bins:
      bin  n  confidence  accuracy
[0.2,0.3)  8    0.250996  0.125000
[0.3,0.4) 28    0.347080  0.178571
[0.4,0.5) 16    0.446497  0.125000
[0.5,0.6) 32    0.545866  0.187500
[0.6,0.7) 19    0.650989  0.105263
[0.7,0.8) 15    0.758078  0.266667
[0.8,0.9) 17    0.864703  0.235294
[0.9,1.0) 75    0.970985  0.573333
cut 2021 bins:
      bin  n  confidence  accuracy
[0.1,0.2)  3    0.187662  0.333333
[0.2,0.3) 25    0.256529  0.160000
[0.3,0.4) 38    0.349389  0.184211
[0.4,0.5) 33    0.451776  0.090909
[0.5,0.6) 40    0.551626  0.150000
[0.6,0.7) 38    0.644771  0.263158
[0.7,0.8) 20    0.737836  0.100000
[0.8,0.9) 28    0.850031  0.178571
[0.9,1.0) 

## 7. Ket luan theo dung quy tac da chot o dau notebook

Doc theo bang quy tac, khong doc theo an tuong.

In [10]:
# The conditional is called changed when the across-window agreement falls below
# the 2.5th percentile of the within-window control, and the class-proportion
# matched statistic agrees. Comparing to the control's own null avoids a chosen
# cut-off; requiring the matched statistic too means a change in class
# composition alone cannot produce the verdict.
verdicts = []
for cut in sorted(windows):
    if cut not in SURVIVING:
        verdicts.append({"cut": cut, "input_shift": "not tested",
                         "conditional_shift": "not tested",
                         "conclusion": "residual not distinguishable from zero"})
        continue
    d = domain[domain["cut"] == cut].iloc[0]
    c = conditional[conditional["cut"] == cut].iloc[0]
    input_shift = {"shift": "yes", "no_shift": "no", "undetermined": "undetermined"}[d.verdict]
    matched_below = (c["across_matched"] == c["across_matched"]
                     and c["across_matched"] < c["within_lo"])
    association_changed = bool(c["below_within_null"]) and matched_below

    # The statistic above summarises the marginal feature-label association,
    # which is a statement about p(x|y). Covariate shift moves p(x|y) exactly as
    # concept drift does, so no value of it is evidence about p(y|x). See the
    # counterexample in the next section. Nothing in this study observes
    # p(y|x), so the conditional column is "no evidence" by construction and
    # not because the statistic came out small.
    conditional_shift = "no evidence"

    if input_shift == "yes" and conditional_shift == "yes":
        conclusion = "concept drift"
    elif input_shift == "yes" and conditional_shift == "no evidence":
        conclusion = "covariate shift, NOT concept drift"
    elif input_shift == "no" and conditional_shift == "no evidence":
        conclusion = "unexplained residual, source outside the taxonomy"
    else:
        conclusion = "unexplained residual, evidence not clear"
    verdicts.append({"cut": cut, "input_shift": input_shift,
                     "conditional_shift": conditional_shift,
                     "association_changed": association_changed,
                     "conclusion": conclusion})
verdict_table = pd.DataFrame(verdicts)
display(verdict_table)
print(verdict_table["conclusion"].value_counts().to_string())
print()
print("The association statistic, reported for what it does establish:")
for row in conditional[conditional["in_scope"]].itertuples():
    print(f"  cut {row.cut}: across={row.across:+.3f} matched={row.across_matched:+.3f} "
          f"within=[{row.within_lo:+.3f},{row.within_hi:+.3f}] "
          f"below_null={row.below_within_null}")

,cut,input_shift,conditional_shift,association_changed,conclusion
0,2015,yes,no evidence,True,"covariate shift, NOT concept drift"
1,2017,not tested,not tested,NaN,residual not distinguishable from zero
2,2019,yes,no evidence,True,"covariate shift, NOT concept drift"
3,2021,yes,no evidence,True,"covariate shift, NOT concept drift"
4,2023,not tested,not tested,NaN,residual not distinguishable from zero


conclusion
covariate shift, NOT concept drift        3
residual not distinguishable from zero    2

The association statistic, reported for what it does establish:
  cut 2015: across=+0.006 matched=+0.042 within=[+0.329,+0.448] below_null=True
  cut 2019: across=+0.322 matched=+0.340 within=[+0.496,+0.632] below_null=True
  cut 2021: across=+0.335 matched=+0.405 within=[+0.533,+0.629] below_null=True


## 8. Tuong quan giua $R$ va phep do drift doc lap (muc 8.6)

Co tuong quan la bang chung that cho viec $R$ mang tin hieu drift. **Khong** co
tuong quan cung la ket qua dang bao cao, khong duoc giau.

Luu y ve suc manh: phep do drift la mot gia tri cho moi cut, con $R$ co 20 o, nen
thuc chat chi co 5 gia tri $x$ phan biet duoc. Tuong quan nay tu no da yeu; bao he
so kem p-value va **khong** duoc doc ket qua rong thanh bang chung cho viec khong
co drift.

In [11]:
# A second, classifier-free distance: the mean absolute standardised difference
# in feature means between the two windows.
distances = []
for cut, window in windows.items():
    pooled = np.vstack([window["x_early"], window["x_late"]])
    scale = pooled.std(axis=0)
    scale[scale == 0.0] = np.inf
    gap = np.abs(window["x_late"].mean(axis=0) - window["x_early"].mean(axis=0)) / scale
    distances.append({"cut": cut, "mean_abs_std_diff": float(gap.mean())})
distances = pd.DataFrame(distances).merge(domain[["cut", "excess"]], on="cut")
display(distances)

merged = residual_ci.merge(distances, on="cut")
for column in ("excess", "mean_abs_std_diff"):
    outcome = correlate_residual_with_drift(merged["R"].to_numpy(),
                                            merged[column].to_numpy())
    print(f"R vs {column:<20} rho={outcome['statistic']:+.3f}  "
          f"p={outcome['p_value']:.4f}  n={outcome['n_cells']} cells "
          f"({merged['cut'].nunique()} distinct cut values)")

,cut,mean_abs_std_diff,excess
0,2015,0.195813,0.300000
1,2017,0.252840,0.406250
2,2019,0.207915,0.304762
3,2021,0.185816,0.339506
4,2023,0.240977,0.333333


R vs excess               rho=+0.294  p=0.2078  n=20 cells (5 distinct cut values)
R vs mean_abs_std_diff    rho=-0.251  p=0.2850  n=20 cells (5 distinct cut values)


## 9. Cau chu bat buoc cho ban thao

> $R$ la can tren cho phan dong gop cua drift, khong phai mot uoc luong cua
> concept drift. No hap thu moi nguon chua duoc kiem soat, trong do co chenh lech
> co tap huan luyen o cac cut som va sai so nhan ho.

Cau nay phai xuat hien trong phan ket qua, khong phai chi trong phan gioi han.

In [12]:
out = RESULTS_DIR / "shapley_e3"
PROTECTED = {"metrics.parquet", "predictions.parquet", "run_manifest.json"}
assert out.name not in PROTECTED and out != RESULTS_DIR
out.mkdir(parents=True, exist_ok=True)
residual_ci.to_parquet(out / "residual_ci.parquet", index=False)
domain.to_parquet(out / "domain_classifier.parquet", index=False)
conditional.to_parquet(out / "conditional_structure.parquet", index=False)
verdict_table.to_parquet(out / "residual_verdict.parquet", index=False)
print("wrote:", sorted(f.name for f in out.glob("*.parquet")))
print("run outputs untouched:", sorted(f.name for f in RESULTS_DIR.glob("*.parquet")))

wrote: ['agreement_report.parquet', 'coalition_values_as_reported.parquet', 'coalition_values_as_reported_accuracy.parquet', 'coalition_values_as_reported_macro_f1.parquet', 'coalition_values_common.parquet', 'coalition_values_common_accuracy.parquet', 'coalition_values_common_macro_f1.parquet', 'conditional_structure.parquet', 'decomposition_as_reported.parquet', 'decomposition_as_reported_accuracy.parquet', 'decomposition_as_reported_macro_f1.parquet', 'decomposition_common.parquet', 'decomposition_common_accuracy.parquet', 'decomposition_common_macro_f1.parquet', 'domain_classifier.parquet', 'model_summary.parquet', 'residual_ci.parquet', 'residual_verdict.parquet']
run outputs untouched: ['metrics.parquet', 'predictions.parquet']
